Reference: https://github.com/medspacy/medspacy/blob/master/notebooks/03-Information-Extraction.ipynb

In [1]:
import spacy
import medspacy

# Option 1: Load default
nlp = medspacy.load()

In [2]:
# Option 2: Load from existing model
nlp = spacy.load("en_core_web_sm", disable={"ner"})
nlp = medspacy.load(nlp)

In [3]:
# Option 3: Load from model name
nlp = medspacy.load("en_core_web_sm", disable={"ner"})

In [4]:
import medspacy
from medspacy.ner import TargetRule
from medspacy.visualization import visualize_ent

# Load medspacy model
nlp = medspacy.load()
print(nlp.pipe_names)

text = """
Past Medical History:
1. Atrial fibrillation
2. Type II Diabetes Mellitus

Assessment and Plan:
There is no evidence of pneumonia. Continue warfarin for Afib. Follow up for management of type 2 DM.
"""

# Add rules for target concept extraction
target_matcher = nlp.get_pipe("medspacy_target_matcher")
target_rules = [
    TargetRule("atrial fibrillation", "PROBLEM"),
    TargetRule("atrial fibrillation", "PROBLEM", pattern=[{"LOWER": "afib"}]),
    TargetRule("pneumonia", "PROBLEM"),
    TargetRule("Type II Diabetes Mellitus", "PROBLEM", 
              pattern=[
                  {"LOWER": "type"},
                  {"LOWER": {"IN": ["2", "ii", "two"]}},
                  {"LOWER": {"IN": ["dm", "diabetes"]}},
                  {"LOWER": "mellitus", "OP": "?"}
              ]),
    TargetRule("warfarin", "MEDICATION")
]
target_matcher.add(target_rules)

doc = nlp(text)
visualize_ent(doc)

['medspacy_pyrush', 'medspacy_target_matcher', 'medspacy_context']


In [5]:
import medspacy
from medspacy.ner import TargetRule
from medspacy.visualization import visualize_ent
from spacy.tokens import Span

# Load medspacy model
nlp = medspacy.load()
print(nlp.pipe_names)

Span.set_extension("icd10_code", default="")

text = """
DIAGNOSIS: Type II Diabetes Mellitus
The patient presents today for management of Type 2 DM.
"""

# Add rules for target concept extraction
target_matcher = nlp.get_pipe("medspacy_target_matcher")
target_rules = [
    TargetRule(
        literal="Type II Diabetes Mellitus",
        category="PROBLEM",
        attributes={"icd10_code": "E11.9"}
    ),
    TargetRule(
        literal="Type II Diabetes Mellitus",
        category="PROBLEM",
        pattern=[{"LOWER": "type"}, {"LOWER": {"IN": ["two", "ii", "2"]}}, {"LOWER": "dm"}],
        attributes={"icd10_code": "E11.9"}
    ),
]
target_matcher.add(target_rules)

doc = nlp(text)
visualize_ent(doc)

['medspacy_pyrush', 'medspacy_target_matcher', 'medspacy_context']


In [6]:
# Even though different rules were used to match the ents,
# they have the same 'literal' value, and both are assigned "E11.9" 
# as an icd10 code
for ent in doc.ents:
    print(ent, ent._.target_rule.literal, ent._.icd10_code, sep="\t")

Type II Diabetes Mellitus	Type II Diabetes Mellitus	E11.9
Type 2 DM	Type II Diabetes Mellitus	E11.9
